# Пайплайн 1. Региональные представления метилома для эпигенетических часов: реальные данные

Ноутбук реализует на реальных открытых данных RRBS крови мыши (mm10) эксперименты первого плана продолжения
проекта: как отбор CpG, агрегация, описание внутреннего паттерна, границы по сигналу (оригинальный MethyLasso),
границы состояний хроматина и пространственные контакты Hi-C влияют на точность и переносимость часов возраста.

| Блок | Вопрос |
|---|---|
| E1 | Что даёт выигрыш: отбор сайтов (A→B), агрегация на тех же сайтах (B→C1), использование всех наблюдаемых CpG кластера (C1→C2) или паттерн (C2→D) |
| E2 | Какой компонент описания региона полезен (mean, sd, slope, enddiff) при контроле числа признаков (bins4) |
| E3 | DBSCAN против сегментации по сигналу (MethyLasso), гибрида и окон 5 кб (RegBC) |
| E4 | Регионы ChromHMM и пересечение DBSCAN ∩ ChromHMM, со средним и с паттерном |
| E5 | Графовый штраф по контактам Hi-C против линейного графа, рандомизированного графа и графа другого типа клеток |
| E6 | Устойчивость к удалению CpG и прореживанию чтений на внешней когорте |
| CR | Ускорение возраста у мышей на ограничении калорий |

**Данные.** Обучение и внутренний тест: GSE80672 (Petkovich et al., 2017; mm10-переобработка GSE120136).
Внешний тест: GSE120132 (Thompson et al., 2018), кровь C57BL/6J и BALB/cByJ. Аннотации: full-stack ChromHMM mm10
(Vu & Ernst, 2023); Hi-C CH12 и mESC из 4DN. Сегментация: MethyLasso (Balaramane et al., 2024), исходный код авторов.

**Требования.** Google Colab (стандартная среда, 12 ГБ RAM); около 10 ГБ временного диска; полный прогон
занимает порядка 1–1,5 ч (основное время — загрузка ~9 ГБ и обучение LASSO на ~10⁶ CpG в ветке A).
Выполняйте ячейки по порядку: «Среда выполнения → Выполнить все».

**Ограничения интерпретации.** Это один фиксированный сплит обучающей когорты и одна внешняя когорта той же
лаборатории обработки (BS-Seeker2, UCLA); различия, чьи бутстреп-интервалы ΔMAE включают ноль, не следует
считать установленными. ChromHMM и Hi-C — внешние референсы не из крови, поэтому выводы касаются пользы
референсного контекста, а не индивидуальной 3D-организации.


## 1. Конфигурация
Все пути задаются здесь. По умолчанию в Colab используется `/content/regclock_work`; чтобы не скачивать
данные повторно, можно указать каталог на Google Drive (ячейка ниже монтирует Drive при `USE_DRIVE=True`).

In [ ]:
import os, sys, json, time, subprocess, shutil
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False                      # True — хранить данные и кэш на Google Drive
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/regclock_work"
else:
    WORK = os.environ.get("REGCLOCK_WORK", "/content/regclock_work" if IN_COLAB else os.path.abspath("regclock_work"))
DATA = os.environ.get("REGCLOCK_DATA", os.path.join(WORK, "data"))
OUT = os.path.join(WORK, "results_pipeline1")
CODE = os.path.join(WORK, "code")
for d in (WORK, DATA, OUT, CODE):
    os.makedirs(d, exist_ok=True)
sys.path.insert(0, CODE)

SEED = 20261014          # фиксированное зерно для разбиения и всех случайных процедур
MIN_COV = 5              # минимальное покрытие CpG (как в RegBC)
MIN_FRAC_TRAIN = 0.5     # словарь CpG: покрытие >= MIN_COV у >= 50% обучающих образцов
QC_FRAC_CPG = 0.9        # ветка A: отдельные CpG с покрытием >= MIN_COV у >= 90% обучающих образцов
DBSCAN_EPS, DBSCAN_MINPTS = 1000, 5   # параметры eps1000 DRegBC (Simpson et al., 2023)
RUN_METHYLASSO = True    # оригинальный MethyLasso (R) из github.com/bardetlab/methylasso
RUN_HIC = True           # загрузка Hi-C из 4DN (CH12 ~4.5 ГБ, mESC ~1.5 ГБ; после извлечения удаляются)
print("WORK =", WORK)

## 2. Установка зависимостей
Python: `cooler` для чтения `.mcool`. R: зависимости MethyLasso ставятся бинарными пакетами
Posit Package Manager (быстро в Colab). В `src/Makevars` MethyLasso стандарт C++11 заменяется на C++17:
новые заголовки Boost (пакет BH) не компилируются в режиме C++11. Алгоритм не изменяется.

In [ ]:
def sh(cmd, check=True):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.stdout.strip():
        print(r.stdout[-3000:])
    if r.returncode != 0:
        print(r.stderr[-3000:])
        if check:
            raise RuntimeError(f"Команда завершилась с кодом {r.returncode}")
    return r

sh(f"{sys.executable} -m pip install -q cooler pyarrow scikit-learn scipy pandas numpy matplotlib")

METHYLASSO_DIR = os.path.join(WORK, "methylasso")
def install_methylasso():
    if shutil.which("Rscript") is None:
        sh("apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq r-base-core")
    ok = sh("Rscript -e 'quit(status = !requireNamespace(\"MethyLasso\", quietly=TRUE))'", check=False)
    if ok.returncode == 0:
        print("MethyLasso уже установлен"); return
    sh("apt-get install -y -qq libgsl-dev > /dev/null 2>&1 || true", check=False)
    codename = subprocess.run("lsb_release -cs", shell=True, text=True, capture_output=True).stdout.strip() or "jammy"
    pkgs = 'c("Rcpp","RcppEigen","BH","RcppGSL","matrixStats","scales","Matrix","data.table","ggplot2","foreach","doParallel","stringr","R.utils","optparse")'
    sh(f"""Rscript -e 'options(repos=c(CRAN="https://packagemanager.posit.co/cran/__linux__/{codename}/latest"), HTTPUserAgent=sprintf("R/%s R (%s)", getRversion(), paste(getRversion(), R.version$platform, R.version$arch, R.version$os))); need <- setdiff({pkgs}, rownames(installed.packages())); if (length(need)) install.packages(need, Ncpus=2)'""")
    if not os.path.exists(METHYLASSO_DIR):
        sh(f"git clone -q https://github.com/bardetlab/methylasso.git {METHYLASSO_DIR}")
    sh(f"cd {METHYLASSO_DIR} && git rev-parse HEAD > COMMIT && sed -i 's/CXX_STD = CXX11/CXX_STD = CXX17/' src/Makevars")
    sh(f"R CMD INSTALL --preclean {METHYLASSO_DIR} > {WORK}/methylasso_install.log 2>&1 || (tail -30 {WORK}/methylasso_install.log; false)")

if RUN_METHYLASSO:
    install_methylasso()
    if not os.path.exists(os.path.join(METHYLASSO_DIR, "MethyLasso.R")):
        sh(f"git clone -q https://github.com/bardetlab/methylasso.git {METHYLASSO_DIR}")
    sh(f"Rscript {METHYLASSO_DIR}/MethyLasso.R --version")

## 3. Код модулей
Ячейки ниже записывают три модуля в `WORK/code`: сборка когорт (`data_build.py`), регионы/признаки/модели
(`core.py`) и извлечение контактов Hi-C (`hic_extract.py`). Код полностью приведён в ноутбуке.

In [ ]:
%%writefile {CODE}/data_build.py
"""Сборка реальных когорт RRBS крови мыши (mm10) для обоих пайплайнов.

Источники (NCBI GEO, открытый доступ):
  * GSE80672 (Petkovich et al., 2017) — кровь C57BL/6, переобработка в mm10
    из GSE120136 (BS-Seeker2, файлы CGfinal), используется как обучающая когорта;
  * GSE120132 (Thompson et al., 2018) — кровь C57BL/6J и BALB/cByJ, те же CGfinal mm10,
    используется как независимая внешняя когорта.
Формат CGfinal (описание GSE120136): chromosome, dinucleotide, position,
methylated counts, total counts, methylation frequency, 95% CI.
"""
import gzip, io, os, re, subprocess, time, urllib.request
import numpy as np
import pandas as pd

AUTOSOMES = [f"chr{i}" for i in range(1, 20)]
CHR_INDEX = {c: i for i, c in enumerate(AUTOSOMES)}
GEO = "https://ftp.ncbi.nlm.nih.gov/geo"
PETKOVICH_TAR = f"{GEO}/series/GSE120nnn/GSE120136/suppl/GSE120136_GSE80672_re-analyzed_Samples_processed_data.tar.gz"
PETKOVICH_MATRIX = f"{GEO}/series/GSE80nnn/GSE80672/matrix/GSE80672_series_matrix.txt.gz"
THOMPSON_MATRICES = [f"{GEO}/series/GSE120nnn/GSE120132/matrix/GSE120132-GPL17021_series_matrix.txt.gz",
                     f"{GEO}/series/GSE120nnn/GSE120132/matrix/GSE120132-GPL21103_series_matrix.txt.gz"]
THOMPSON_FILELIST = f"{GEO}/series/GSE120nnn/GSE120132/suppl/filelist.txt"


def _get(url, dest):
    if not os.path.exists(dest) or os.path.getsize(dest) == 0:
        subprocess.run(["curl", "-sSfL", "--retry", "5", "-o", dest, url], check=True)
    return dest


def parse_series_matrix(path):
    rows = {}
    with gzip.open(path, "rt", errors="replace") as fh:
        for line in fh:
            if not line.startswith("!Sample_"):
                continue
            parts = [x.strip().strip('"') for x in line.rstrip("\n").split("\t")]
            key, vals = parts[0], parts[1:]
            if key == "!Sample_characteristics_ch1":
                name = vals[0].split(":")[0].strip()
                rows[name] = [v.split(":", 1)[1].strip() if ":" in v else v for v in vals]
            else:
                rows[key.replace("!Sample_", "")] = vals
    n = len(rows["geo_accession"])
    return pd.DataFrame({k: v for k, v in rows.items() if len(v) == n})


def petkovich_metadata(workdir):
    p = _get(PETKOVICH_MATRIX, os.path.join(workdir, "GSE80672_series_matrix.txt.gz"))
    m = parse_series_matrix(p)
    # В GEO поле подписано "age (years)", но значения 0.67–35 соответствуют месяцам
    # (Petkovich et al., 2017: мыши 3–35 мес.). Перевод не выполняется, единица — месяцы.
    m["age_months"] = pd.to_numeric(m["age (years)"], errors="coerce")
    base = (m["tissue"].str.lower() == "whole blood") & (m["strain"] == "C57BL/6") & \
           (m["genetic condition"] == "Wild Type")
    m["group"] = np.where(base & (m["diet"] == "standard"), "petkovich_control",
                  np.where(base & (m["diet"] == "Calorie Restricted"), "petkovich_CR", "excluded"))
    m = m[m.group != "excluded"].copy()
    m["sample_id"] = m["geo_accession"]
    m["cohort"] = "GSE80672"
    m["sex"] = m["gender"]
    return m[["sample_id", "title", "cohort", "group", "age_months", "sex", "strain", "diet"]].reset_index(drop=True)


def thompson_metadata(workdir):
    ms = []
    for u in THOMPSON_MATRICES:
        ms.append(parse_series_matrix(_get(u, os.path.join(workdir, os.path.basename(u)))))
    m = pd.concat(ms, ignore_index=True)
    m = m[m["tissue"] == "Blood"].copy()
    m["age_months"] = m["age"].str.replace("mo", "", regex=False).astype(float)
    m["group"] = np.where(m["strain"] == "C57BL/6J", "thompson_B6", "thompson_BALB")
    m["sample_id"] = m["geo_accession"]
    m["cohort"] = "GSE120132"
    m["sex"] = m["Sex"]
    m["diet"] = "standard"
    return m[["sample_id", "title", "cohort", "group", "age_months", "sex", "strain", "diet"]].reset_index(drop=True)


def download_petkovich(datadir, keep_ids):
    """Потоковая распаковка архива GSE120136 (~2.4 ГБ); на диске остаются только нужные образцы."""
    os.makedirs(datadir, exist_ok=True)
    done = os.path.join(datadir, ".done")
    if not os.path.exists(done):
        cmd = f"curl -sSfL --retry 5 '{PETKOVICH_TAR}' | tar xzf - -C '{datadir}'"
        subprocess.run(cmd, shell=True, check=True)
        open(done, "w").close()
    paths = {}
    for f in os.listdir(datadir):
        mm = re.match(r"GSE80672_(GSM\d+)\.CGfinal\.txt\.gz", f)
        if mm:
            if mm.group(1) in keep_ids:
                paths[mm.group(1)] = os.path.join(datadir, f)
            else:
                os.remove(os.path.join(datadir, f))
    return paths


def download_thompson(datadir, keep_ids, workers=6):
    os.makedirs(datadir, exist_ok=True)
    fl = _get(THOMPSON_FILELIST, os.path.join(datadir, "filelist.txt"))
    names = [l.split("\t")[1] for l in open(fl) if "Mouse_Blood" in l]
    paths, jobs = {}, []
    for n in names:
        gsm = n.split("_")[0]
        if gsm not in keep_ids:
            continue
        dest = os.path.join(datadir, n)
        paths[gsm] = dest
        if not os.path.exists(dest) or os.path.getsize(dest) == 0:
            jobs.append(f"{GEO}/samples/{gsm[:-3]}nnn/{gsm}/suppl/{n} {dest}")
    if jobs:
        lst = os.path.join(datadir, "jobs.txt")
        open(lst, "w").write("\n".join(jobs) + "\n")
        subprocess.run(f"cat '{lst}' | xargs -P {workers} -n 2 sh -c 'curl -sSfL --retry 5 -o \"$1\" \"$0\"'",
                       shell=True, check=True)
    return paths


def read_cgfinal(path, min_cov=1):
    """Возвращает (key, mC, cov) для аутосом; key = chr_index * 2^32 + pos (исходная позиция CGfinal)."""
    d = pd.read_csv(path, sep=r"\s+", header=None, usecols=[0, 2, 3, 4],
                    names=["chr", "pos", "mc", "cov"], engine="c",
                    dtype={"chr": str, "pos": np.int64, "mc": np.int64, "cov": np.int64})
    d = d[d["chr"].isin(CHR_INDEX) & (d["cov"] >= min_cov)]
    if (d["mc"] > d["cov"]).any() or (d["mc"] < 0).any():
        raise ValueError(f"Нарушено 0 <= mC <= coverage в {path}")
    key = d["chr"].map(CHR_INDEX).values.astype(np.int64) * (1 << 32) + d["pos"].values
    if len(np.unique(key)) != len(key):
        raise ValueError(f"Дубликаты координат в {path}")
    return key, d["mc"].values, d["cov"].values


def build_matrix(samples, paths, train_ids, out_npz, min_cov=5, min_frac_train=0.5, log=print):
    """Два прохода: (1) словарь сайтов по обучающим образцам; (2) counts всех образцов на этом словаре.
    Словарь зависит только от покрытия обучающих образцов (без возраста)."""
    if os.path.exists(out_npz):
        z = np.load(out_npz, allow_pickle=True)
        return z["keys"], z["mc"], z["cov"], list(z["sample_ids"])
    t0 = time.time()
    batch_u, batch_c, buf = [], [], []
    def flush():
        if buf:
            u, c = np.unique(np.concatenate(buf), return_counts=True)
            batch_u.append(u); batch_c.append(c); buf.clear()
    for k, sid in enumerate(train_ids):
        key, mc, cov = read_cgfinal(paths[sid])
        buf.append(key[cov >= min_cov])
        if len(buf) == 20:
            flush()
        if (k + 1) % 20 == 0:
            log(f"  проход 1: {k+1}/{len(train_ids)} обучающих образцов, {time.time()-t0:.0f} с")
    flush()
    allu = np.concatenate(batch_u); allc = np.concatenate(batch_c)
    order = np.argsort(allu, kind="stable"); allu, allc = allu[order], allc[order]
    uu, start = np.unique(allu, return_index=True)
    tot = np.add.reduceat(allc, start)
    keys = uu[tot >= min_frac_train * len(train_ids)]
    log(f"  словарь: {len(keys)} CpG с покрытием >= {min_cov} у >= {min_frac_train:.0%} обучающих образцов")
    ids = list(samples)
    MC = np.zeros((len(ids), len(keys)), dtype=np.uint16)
    CV = np.zeros((len(ids), len(keys)), dtype=np.uint16)
    for k, sid in enumerate(ids):
        key, mc, cov = read_cgfinal(paths[sid])
        idx = np.searchsorted(keys, key)
        idx[idx == len(keys)] = 0
        hit = keys[idx] == key
        # Ограничение uint16: при покрытии > 65535 counts масштабируются пропорционально (практически не встречается)
        cv = cov[hit].astype(np.float64); mcv = mc[hit].astype(np.float64)
        scale = np.where(cv > 65535, 65535 / cv, 1.0)
        CV[k, idx[hit]] = np.round(cv * scale).astype(np.uint16)
        MC[k, idx[hit]] = np.round(mcv * scale).astype(np.uint16)
        if (k + 1) % 25 == 0:
            log(f"  проход 2: {k+1}/{len(ids)} образцов, {time.time()-t0:.0f} с")
    np.savez(out_npz, keys=keys, mc=MC, cov=CV, sample_ids=np.array(ids))
    return keys, MC, CV, ids


def key_chr(keys):
    return (keys >> 32).astype(np.int32)


def key_pos(keys):
    return (keys & ((1 << 32) - 1)).astype(np.int64)


In [ ]:
%%writefile {CODE}/core.py
"""Общие функции: регионы, региональные признаки, модель возраста, метрики."""
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.cluster import DBSCAN
from sklearn.linear_model import LassoCV, Lasso, lasso_path
from sklearn.model_selection import KFold
from scipy import stats

MIN_COV = 5


# ----------------------------------------------------------------------------- данные
class Methylome:
    """Обёртка над матрицей counts: образцы × CpG (uint16)."""

    def __init__(self, keys, mc, cov, sample_ids, meta):
        self.keys = keys
        self.chr = (keys >> 32).astype(np.int32)
        self.pos = (keys & ((1 << 32) - 1)).astype(np.int64)
        self.mc = mc
        self.cov = cov
        self.ids = list(sample_ids)
        self.meta = meta.set_index("sample_id").loc[self.ids].reset_index()
        self.row = {s: i for i, s in enumerate(self.ids)}

    def rows(self, split):
        return np.where(self.meta["split"].values == split)[0]

    def beta(self, rows, sites=None, mc=None, cov=None, min_cov=MIN_COV):
        mc = self.mc if mc is None else mc
        cov = self.cov if cov is None else cov
        sites = np.arange(cov.shape[1]) if sites is None else np.asarray(sites)
        rows = np.asarray(rows)
        b = np.full((len(rows), len(sites)), np.nan, dtype=np.float32)
        for j0 in range(0, len(sites), 100000):  # поблочно: пиковая память ~ размер результата
            sj = sites[j0:j0 + 100000]
            c = cov[np.ix_(rows, sj)].astype(np.float32)
            m = mc[np.ix_(rows, sj)].astype(np.float32)
            with np.errstate(invalid="ignore", divide="ignore"):
                b[:, j0:j0 + 100000] = np.where(c >= min_cov, m / c, np.nan)
        return b


def degrade_counts(mc, cov, mode, level, rng):
    """mode='mask': случайно удалить долю `level` наблюдаемых CpG;
    mode='thin': биномиальное прореживание чтений с вероятностью сохранения `level`."""
    mc = mc.astype(np.int64); cov = cov.astype(np.int64)
    if mode == "mask":
        drop = rng.random(cov.shape) < level
        cov = np.where(drop, 0, cov); mc = np.where(drop, 0, mc)
    elif mode == "thin":
        m2 = rng.binomial(mc, level)
        u2 = rng.binomial(cov - mc, level)
        mc, cov = m2, m2 + u2
    return mc.astype(np.uint16), cov.astype(np.uint16)


# ----------------------------------------------------------------------------- регионы
def regions_from_labels(chr_, pos, labels, site_idx):
    """labels >= 0 — номер региона для сайтов site_idx; возвращает таблицу регионов и массив принадлежности."""
    df = pd.DataFrame({"site": site_idx, "chr": chr_[site_idx], "pos": pos[site_idx], "lab": labels})
    df = df[df.lab >= 0]
    g = df.groupby("lab", sort=False)
    reg = pd.DataFrame({"chr": g.chr.first(), "start": g.pos.min(), "end": g.pos.max() + 2, "n_cpg": g.size()})
    reg = reg.sort_values(["chr", "start"]).reset_index()
    remap = pd.Series(np.arange(len(reg)), index=reg["lab"].values)
    member = np.full(len(chr_), -1, dtype=np.int64)
    member[df.site.values] = remap.loc[df.lab.values].values
    return reg.drop(columns="lab"), member


def dbscan_regions(chr_, pos, site_idx, eps=1000, min_samples=5):
    """DBSCAN по геномной координате (как DRegBC в Simpson et al., 2023: eps в п.н., minPts=5)."""
    labels = np.full(len(site_idx), -1, dtype=np.int64)
    offset = 0
    c = chr_[site_idx]
    for ch in np.unique(c):
        w = np.where(c == ch)[0]
        x = pos[site_idx[w]].reshape(-1, 1).astype(np.float64)
        lab = DBSCAN(eps=eps, min_samples=min_samples, algorithm="kd_tree").fit_predict(x)
        ok = lab >= 0
        labels[w[ok]] = lab[ok] + offset
        offset += (lab.max() + 1) if ok.any() else 0
    return regions_from_labels(chr_, pos, labels, site_idx)


def window_regions(chr_, pos, site_idx, width=5000, min_cpg=1):
    """Скользящие (непересекающиеся) окна фиксированной длины, как RegBC 5 kb."""
    lab = chr_[site_idx].astype(np.int64) * 10**7 + pos[site_idx] // width
    u, inv, cnt = np.unique(lab, return_inverse=True, return_counts=True)
    inv = np.where(cnt[inv] >= min_cpg, inv, -1)
    return regions_from_labels(chr_, pos, inv, site_idx)


def interval_regions(chr_, pos, site_idx, intervals, min_cpg=3):
    """Регионы по внешним интервалам (BED, 0-based полуоткрытые) — например, MethyLasso или ChromHMM.
    intervals: DataFrame chr(int-индекс), start, end, неперекрывающиеся."""
    labels = np.full(len(site_idx), -1, dtype=np.int64)
    c = chr_[site_idx]; p = pos[site_idx]
    for ch, iv in intervals.groupby("chr"):
        iv = iv.sort_values("start")
        w = np.where(c == ch)[0]
        if len(w) == 0:
            continue
        # CGfinal-позиции трактуются как 1-based координата C; BED-интервал [start, end) 0-based
        p0 = p[w] - 1
        k = np.searchsorted(iv.start.values, p0, side="right") - 1
        ok = (k >= 0) & (p0 < iv.end.values[np.clip(k, 0, None)])
        labels[w[ok]] = iv.index.values[k[ok]]
    reg, member = regions_from_labels(chr_, pos, labels, site_idx)
    keep = reg.n_cpg.values >= min_cpg
    newid = np.full(len(reg), -1); newid[keep] = np.arange(keep.sum())
    member = np.where(member >= 0, newid[np.clip(member, 0, None)], -1)
    return reg[keep].reset_index(drop=True), member


def intersect_partitions(member_a, member_b, chr_, pos, min_cpg=3):
    """Пересечение двух разбиений: сайт остаётся в регионе, только если он принадлежит обоим."""
    ok = (member_a >= 0) & (member_b >= 0)
    lab = np.where(ok, member_a.astype(np.int64) * 10**7 + member_b, -1)
    site_idx = np.where(ok)[0]
    u, inv = np.unique(lab[site_idx], return_inverse=True)
    reg, member = regions_from_labels(chr_, pos, inv, site_idx)
    keep = reg.n_cpg.values >= min_cpg
    newid = np.full(len(reg), -1); newid[keep] = np.arange(keep.sum())
    member = np.where(member >= 0, newid[np.clip(member, 0, None)], -1)
    return reg[keep].reset_index(drop=True), member


# ----------------------------------------------------------------------------- признаки
def _assign(member, xnorm=None, nbins=None):
    sites = np.where(member >= 0)[0]
    nreg = member.max() + 1
    if nbins is None:
        cols = member[sites]; ncol = nreg
    else:
        b = np.minimum((xnorm[sites] * nbins).astype(np.int64), nbins - 1)
        cols = member[sites] * nbins + b; ncol = nreg * nbins
    A = sp.csr_matrix((np.ones(len(sites), np.float32), (np.arange(len(sites)), cols)), shape=(len(sites), ncol))
    return sites, A


def relative_position(member, pos):
    """Нормированная координата CpG внутри региона: 0 — первый CpG, 1 — последний."""
    x = np.zeros(len(member), np.float32)
    s = np.where(member >= 0)[0]
    m = member[s]
    mn = pd.Series(pos[s]).groupby(m).transform("min").values
    mx = pd.Series(pos[s]).groupby(m).transform("max").values
    x[s] = (pos[s] - mn) / np.maximum(mx - mn, 1)
    return x


def region_features(M, rows, member, kinds=("mean",), nbins=4, mc=None, cov=None, chunk=40, min_obs_pattern=3):
    """Признаки региона по наблюдаемым CpG (покрытие >= MIN_COV) каждого образца.
    mean — среднее beta; sd — стандартное отклонение между CpG; slope — наклон beta по нормированной
    координате; enddiff — среднее последней трети минус среднее первой трети; bins — средние nbins
    последовательных бинов. Пустые значения = NaN (далее импутируются медианой train)."""
    x = relative_position(member, M.pos)
    sites, A = _assign(member)
    xs = x[sites]
    Ax = sp.csr_matrix(A.multiply(xs[:, None])); Ax2 = sp.csr_matrix(A.multiply((xs ** 2)[:, None]))
    first = sp.csr_matrix(A.multiply((xs <= 1 / 3)[:, None].astype(np.float32)))
    last = sp.csr_matrix(A.multiply((xs >= 2 / 3)[:, None].astype(np.float32)))
    if "bins" in kinds:
        _, Ab = _assign(member, x, nbins)
    out = {k: [] for k in kinds}
    for s0 in range(0, len(rows), chunk):
        r = rows[s0:s0 + chunk]
        B = M.beta(r, sites, mc=mc, cov=cov)
        O = np.isfinite(B).astype(np.float32)
        B0 = np.nan_to_num(B)
        n = O @ A; sb = B0 @ A
        with np.errstate(invalid="ignore", divide="ignore"):
            mean = sb / n
            if "mean" in kinds:
                out["mean"].append(mean)
            if "sd" in kinds:
                sb2 = (B0 ** 2) @ A
                var = sb2 / n - mean ** 2
                sd = np.sqrt(np.clip(var * n / np.maximum(n - 1, 1), 0, None))
                sd[n < min_obs_pattern] = np.nan
                out["sd"].append(sd)
            if "slope" in kinds:
                sx = O @ Ax; sx2 = O @ Ax2; sxb = B0 @ Ax
                den = n * sx2 - sx ** 2
                slope = (n * sxb - sx * sb) / den
                slope[(n < min_obs_pattern) | (np.abs(den) < 1e-6)] = np.nan
                out["slope"].append(slope)
            if "enddiff" in kinds:
                nf = O @ first; nl = O @ last
                d = (B0 @ last) / nl - (B0 @ first) / nf
                d[(nf < 1) | (nl < 1)] = np.nan
                out["enddiff"].append(d)
            if "bins" in kinds:
                nb = O @ Ab
                out["bins"].append((B0 @ Ab) / nb)
    return {k: np.vstack(v).astype(np.float32) for k, v in out.items()}


def cpg_features(M, rows, sites, mc=None, cov=None):
    return M.beta(rows, sites, mc=mc, cov=cov)


# ----------------------------------------------------------------------------- модель
class Preprocessor:
    """Импутация медианой train и стандартизация по train; признаки без наблюдений в train удаляются."""

    def fit(self, X):
        p = X.shape[1]
        self.med = np.full(p, np.nan, np.float32); self.mu = np.zeros(p, np.float32); self.sd = np.zeros(p, np.float32)
        for j0 in range(0, p, 50000):
            blk = np.array(X[:, j0:j0 + 50000], dtype=np.float32)
            with np.errstate(all="ignore"):
                import warnings
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", RuntimeWarning)
                    med = np.nanmedian(blk, axis=0)
            bad = ~np.isfinite(blk)
            blk[bad] = np.broadcast_to(med, blk.shape)[bad]
            self.med[j0:j0 + 50000] = med; self.mu[j0:j0 + 50000] = blk.mean(0); self.sd[j0:j0 + 50000] = blk.std(0)
        self.keep = np.isfinite(self.med) & (self.sd > 1e-8)
        return self

    def transform(self, X):
        # одна копия: отбор столбцов, затем импутация и стандартизация на месте (экономия памяти при ~10^6 CpG)
        Z = np.array(X[:, self.keep], dtype=np.float32, order="F")
        med, mu, sd = self.med[self.keep], self.mu[self.keep], self.sd[self.keep]
        for j0 in range(0, Z.shape[1], 50000):
            blk = Z[:, j0:j0 + 50000]
            bad = ~np.isfinite(blk)
            if bad.any():
                blk[bad] = np.broadcast_to(med[j0:j0 + 50000], blk.shape)[bad]
            blk -= mu[j0:j0 + 50000]; blk /= sd[j0:j0 + 50000]
        return Z

    def missing_fraction(self, X):
        return float(np.mean(~np.isfinite(X[:, self.keep])))


class AgeModel:
    """LASSO; alpha выбирается 5-кратной CV внутри train (одинаково для всех представлений)."""

    def __init__(self, seed=0, n_alphas=40, cv=5):
        self.seed, self.n_alphas, self.cv = seed, n_alphas, cv

    def fit(self, X, y):
        self.pre = Preprocessor().fit(X)
        Z = self.pre.transform(X)
        self.ymu = y.mean()
        self.m = LassoCV(alphas=self.n_alphas, cv=KFold(self.cv, shuffle=True, random_state=self.seed),
                         max_iter=20000, tol=1e-4, n_jobs=1, copy_X=False, random_state=self.seed).fit(Z, y)
        self.n_features = int(self.pre.keep.sum())
        self.n_selected = int((self.m.coef_ != 0).sum())
        return self

    def predict(self, X):
        return self.m.predict(self.pre.transform(X))

    def feature_missing(self, X):
        """Доля пропусков среди признаков с ненулевым коэффициентом."""
        idx = np.where(self.pre.keep)[0][self.m.coef_ != 0]
        return float(np.mean(~np.isfinite(X[:, idx]))) if len(idx) else np.nan


def metrics(y, p, n_boot=2000, seed=0):
    y = np.asarray(y, float); p = np.asarray(p, float)
    err = np.abs(p - y)
    rng = np.random.default_rng(seed)
    bs = [err[rng.integers(0, len(err), len(err))].mean() for _ in range(n_boot)]
    ss = ((y - y.mean()) ** 2).sum()
    return {"n": len(y), "MAE": err.mean(), "MAE_lo": np.percentile(bs, 2.5), "MAE_hi": np.percentile(bs, 97.5),
            "MedAE": np.median(err), "R2": 1 - ((y - p) ** 2).sum() / ss, "r": np.corrcoef(y, p)[0, 1],
            "bias": (p - y).mean()}


def paired_mae_diff(y, p1, p2, n_boot=2000, seed=0):
    """Бутстреп разности MAE(p1) - MAE(p2) на одних и тех же образцах."""
    e1 = np.abs(p1 - y); e2 = np.abs(p2 - y); rng = np.random.default_rng(seed)
    d = [(e1[i] - e2[i]).mean() for i in (rng.integers(0, len(y), len(y)) for _ in range(n_boot))]
    return float((e1 - e2).mean()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))


def age_acceleration(pred_ctrl, age_ctrl, pred_case, age_case):
    """Остатки относительно линии pred ~ age, оценённой на контролях вне обучения."""
    b = np.polyfit(age_ctrl, pred_ctrl, 1)
    rc = pred_ctrl - np.polyval(b, age_ctrl); rk = pred_case - np.polyval(b, age_case)
    u = stats.mannwhitneyu(rk, rc)
    return {"median_resid_case": float(np.median(rk)), "median_resid_ctrl": float(np.median(rc)),
            "MWU_p": float(u.pvalue), "n_case": len(rk), "n_ctrl": len(rc)}


# ----------------------------------------------------------------------------- графовый штраф
def region_bins(reg, res=25000):
    return ((reg.start.values + reg.end.values) // 2 // res) * res


def contact_edges(reg, contacts, res=25000, nodes=None):
    """Рёбра между регионами, чьи 25-кб бины имеют контакт (O/E >= порога при извлечении).
    contacts: chr(int), start1, start2, oe. Вес ребра = log2(O/E)."""
    nodes = np.arange(len(reg)) if nodes is None else np.asarray(nodes)
    nd = pd.DataFrame({"node": np.arange(len(nodes)), "chr": reg.chr.values[nodes], "bin": region_bins(reg, res)[nodes]})
    e = contacts.merge(nd.rename(columns={"bin": "start1", "node": "a"}), on=["chr", "start1"]) \
                .merge(nd.rename(columns={"bin": "start2", "node": "b"}), on=["chr", "start2"])
    e = e[e.a != e.b]
    return pd.DataFrame({"a": e.a.values, "b": e.b.values, "w": np.log2(e.oe.values).astype(np.float32),
                         "chr": e.chr.values})


def linear_edges(reg, nodes, max_gap=100000):
    r = reg.iloc[nodes].reset_index(drop=True)
    o = np.lexsort((r.start.values, r.chr.values))
    a, b = o[:-1], o[1:]
    ok = (r.chr.values[a] == r.chr.values[b]) & (r.start.values[b] - r.end.values[a] <= max_gap)
    return pd.DataFrame({"a": a[ok], "b": b[ok], "w": np.ones(ok.sum(), np.float32), "chr": r.chr.values[a[ok]]})


def rewire_edges(edges, seed=0):
    """Рандомизация с сохранением степеней узлов: перестановка вторых концов рёбер внутри хромосомы."""
    rng = np.random.default_rng(seed)
    out = []
    for ch, e in edges.groupby("chr"):
        b = rng.permutation(e.b.values)
        out.append(pd.DataFrame({"a": e.a.values, "b": b, "w": e.w.values, "chr": ch}))
    e = pd.concat(out, ignore_index=True)
    e = e[e.a != e.b]
    return e


def incidence(edges, p):
    m = len(edges)
    if m == 0:
        return sp.csr_matrix((0, p))
    sw = np.sqrt(np.clip(edges.w.values, 0, None)).astype(np.float64)
    rows = np.repeat(np.arange(m), 2)
    cols = np.column_stack([edges.a.values, edges.b.values]).ravel()
    vals = np.column_stack([sw, -sw]).ravel()
    return sp.csr_matrix((vals, (rows, cols)), shape=(m, p))


def _graph_path(Z, yc, B, lam2, alphas):
    Xa = sp.vstack([sp.csr_matrix(Z), np.sqrt(lam2) * B]).tocsc() if lam2 > 0 else Z
    ya = np.concatenate([yc, np.zeros(B.shape[0])]) if lam2 > 0 else yc
    # lasso_path масштабирует квадратичную часть на 1/N; приводим alpha к масштабу числа реальных образцов
    N = Xa.shape[0]; scale = len(yc) / N
    _, coefs, _ = lasso_path(Xa, ya, alphas=alphas * scale, max_iter=5000, tol=1e-3)
    return coefs  # p × n_alphas


def graph_lasso_cv(Z, y, B, lam2_grid, n_alphas=30, cv=5, seed=0):
    """Выбор (lambda2, alpha) 5-кратной CV на train; Z — стандартизованные признаки train."""
    yc = y - y.mean()
    amax = np.abs(Z.T @ yc).max() / len(y)
    alphas = amax * np.logspace(0, -3, n_alphas)
    folds = list(KFold(cv, shuffle=True, random_state=seed).split(Z))
    res = []
    for lam2 in lam2_grid:
        mse = np.zeros(n_alphas)
        for tr, va in folds:
            mu = Z[tr].mean(0); ytr = y[tr]
            coefs = _graph_path(Z[tr] - mu, ytr - ytr.mean(), B, lam2, alphas)
            pred = (Z[va] - mu) @ coefs + ytr.mean()
            mse += ((pred - y[va][:, None]) ** 2).mean(0) / cv
        k = int(np.argmin(mse)); res.append((mse[k], lam2, alphas[k]))
    best = min(res, key=lambda t: t[0])
    mu = Z.mean(0)
    coef = _graph_path(Z - mu, yc, B, best[1], np.array([best[2]]))[:, 0]
    return {"coef": coef, "mu": mu, "ymu": y.mean(), "lam2": best[1], "alpha": best[2],
            "cv_mse": best[0], "grid": res}


In [ ]:
%%writefile {CODE}/hic_extract.py
import cooler, numpy as np, pandas as pd, sys, time
def extract_contacts(mcool, res=25000, min_d_bins=2, max_d_bins=80, oe_min=2.0, raw_min=5, chroms=None, block=400):
    c = cooler.Cooler(f"{mcool}::resolutions/{res}")
    chroms = chroms or [f"chr{i}" for i in range(1, 20)]
    out = []
    for ch in chroms:
        t = time.time()
        ext = c.extent(ch); nb = ext[1]-ext[0]
        w = c.bins().fetch(ch)['weight'].values
        valid = np.isfinite(w)
        parts = []
        for s in range(0, nb, block):
            e = min(nb, s+block); e2 = min(nb, e+max_d_bins)
            px = c.pixels(join=False).fetch((ch, s*res, min(e*res, int(c.chromsizes[ch]))))  # rows in [s,e)
            px = px[(px.bin2_id - ext[0] < e2)]
            i = px.bin1_id.values - ext[0]; j = px.bin2_id.values - ext[0]
            d = j - i
            keep = (d >= min_d_bins) & (d <= max_d_bins) & valid[i] & valid[j]
            i, j, d, cnt = i[keep], j[keep], d[keep], px['count'].values[keep]
            bal = cnt * w[i] * w[j]
            parts.append(pd.DataFrame({'i': i, 'j': j, 'd': d, 'raw': cnt, 'bal': bal}))
        P = pd.concat(parts, ignore_index=True)
        # expected per diagonal = sum(balanced) / number of valid pairs at that distance
        vcum = np.concatenate([[0], np.cumsum(valid)])
        npairs = {dd: int(sum(valid[k] and valid[k+dd] for k in range(0, nb-dd))) for dd in range(min_d_bins, max_d_bins+1)}
        exp = P.groupby('d')['bal'].sum() / pd.Series(npairs)
        P['oe'] = P['bal'] / exp.reindex(P['d']).values
        P = P[(P.oe >= oe_min) & (P.raw >= raw_min)]
        out.append(pd.DataFrame({'chr': ch, 'start1': P.i.values*res, 'start2': P.j.values*res,
                                 'raw': P.raw.values.astype(np.int32), 'oe': P.oe.values.astype(np.float32)}))
        print(ch, nb, len(P), round(time.time()-t, 1), flush=True)
    return pd.concat(out, ignore_index=True)
if __name__ == '__main__':
    df = extract_contacts(sys.argv[1]); df.to_parquet(sys.argv[2]); print(len(df))


## 4. Реальные данные: метаданные, загрузка, разбиение
* **Обучение и внутренний тест** — контрольные мыши C57BL/6 дикого типа на стандартной диете из
  GSE80672 (Petkovich et al., 2017), mm10-переобработка GSE120136. Разбиение 80/20 стратифицировано по
  децилям возраста; один образец соответствует одному животному (повторных измерений в GEO не указано).
* **CR** — мыши C57BL/6 на ограничении калорий из того же GSE80672 (только для оценки ускорения возраста).
* **Внешний тест** — кровь C57BL/6J из GSE120132 (Thompson et al., 2018); дополнительно BALB/cByJ
  (другая линия) как более жёсткий сдвиг.
Словарь CpG строится только по покрытию обучающих образцов, без возраста.

In [ ]:
import numpy as np, pandas as pd
import importlib, data_build as db, core
importlib.reload(db); importlib.reload(core)

meta_p = db.petkovich_metadata(WORK)
meta_t = db.thompson_metadata(WORK)
paths_p = db.download_petkovich(os.path.join(DATA, "petkovich"), set(meta_p.sample_id))
paths_t = db.download_thompson(os.path.join(DATA, "thompson"), set(meta_t.sample_id))
missing = sorted(set(meta_p.sample_id) - set(paths_p))
print(f"Petkovich: {len(meta_p)} в метаданных, {len(paths_p)} файлов в GSE120136; нет файла у {len(missing)}: {missing}")
meta_p = meta_p[meta_p.sample_id.isin(paths_p)]
meta = pd.concat([meta_p, meta_t], ignore_index=True)
paths = {**paths_p, **paths_t}

META_TSV = os.path.join(WORK, "meta.tsv")
if os.path.exists(META_TSV):
    meta = pd.read_csv(META_TSV, sep="\t")
    print("Используется сохранённое разбиение", META_TSV)
else:
    rng = np.random.default_rng(SEED)
    ctrl = meta[meta.group == "petkovich_control"].copy()
    ctrl["stratum"] = pd.qcut(ctrl.age_months.rank(method="first"), 10, labels=False)
    test = []
    for _, g in ctrl.groupby("stratum"):
        test += list(rng.choice(g.sample_id.values, int(round(0.2 * len(g))), replace=False))
    meta["split"] = np.select(
        [meta.sample_id.isin(test), meta.group == "petkovich_control", meta.group == "petkovich_CR",
         meta.group == "thompson_B6"], ["internal_test", "train", "CR", "external_B6"], "external_BALB")
    meta.to_csv(META_TSV, sep="\t", index=False)
print(meta.groupby("split").agg(n=("sample_id", "size"), age_min=("age_months", "min"),
                                age_median=("age_months", "median"), age_max=("age_months", "max"),
                                male=("sex", lambda s: (s == "Male").sum())))

train_ids = meta.sample_id[meta.split == "train"].tolist()
keys, MC, CV, ids = db.build_matrix(meta.sample_id.tolist(), paths, train_ids, os.path.join(WORK, "matrix.npz"),
                                    min_cov=MIN_COV, min_frac_train=MIN_FRAC_TRAIN)
M = core.Methylome(keys, MC, CV, ids, meta)
y = M.meta.age_months.values
SPLITS = ["train", "internal_test", "external_B6", "external_BALB", "CR"]
R = {s: M.rows(s) for s in SPLITS}
print("Матрица:", MC.shape, "CpG в словаре:", len(keys))
obs = pd.DataFrame({s: [(CV[R[s]] >= MIN_COV).mean()] for s in SPLITS}, index=["доля наблюдаемых CpG словаря"])
print(obs.round(3))

## 5. Внешние аннотации: состояния хроматина и контакты Hi-C
* **ChromHMM** — универсальная (pan-tissue) 100-состоянийная full-stack аннотация mm10
  (Vu & Ernst, 2023; github.com/ernstlab/mouse_fullStack_annotations). Соседние сегменты одной группы
  состояний (Prom, TSS, enhancers, ReprPC, HET, quiescent и т.д.) объединяются в один интервал.
  Аннотация не специфична для крови — это ограничение интерпретации.
* **Hi-C** — in situ Hi-C клеток CH12 (B-лимфома мыши, ближайший к крови доступный mm10-референс 4DN,
  4DNFIA7GQD4X) и mESC E14 (4DNFIMN7XO4J) как контроль «другой тип клеток». Из `.mcool` на разрешении
  25 кб извлекаются внутрихромосомные пары бинов на расстоянии 50 кб–2 Мб с балансированным
  O/E >= 2 и сырым числом контактов >= 5; исходный `.mcool` после извлечения удаляется.

In [ ]:
import hic_extract
CI = {f"chr{i+1}": i for i in range(19)}
CHM_DIR = os.path.join(DATA, "chromhmm"); os.makedirs(CHM_DIR, exist_ok=True)
CHM_BASE = "https://public.hoffman2.idre.ucla.edu/ernst/2K9RS//full_stack/full_stack_annotation_public_release/mm10/"
for f in ["mm10_100_segments_segments.bed.gz", "state_annotation_processed.tsv"]:
    db._get(CHM_BASE + f, os.path.join(CHM_DIR, f))
st = pd.read_csv(os.path.join(CHM_DIR, "state_annotation_processed.tsv"), sep="\t")
grp = dict(zip(st.mneumonics, st.group))
chm = pd.read_csv(os.path.join(CHM_DIR, "mm10_100_segments_segments.bed.gz"), sep="\t", header=None,
                  names=["chr", "start", "end", "state"])
chm = chm[chm.chr.isin(CI)].copy()
chm["grp"] = chm.state.str.split("_", n=1).str[1].map(grp)
assert chm.grp.notna().all()
chm["chr"] = chm.chr.map(CI); chm = chm.sort_values(["chr", "start"])
brk = (chm.grp != chm.grp.shift()) | (chm.chr != chm.chr.shift()) | (chm.start != chm.end.shift())
chm["blk"] = brk.cumsum()
chm_blocks = chm.groupby("blk").agg(chr=("chr", "first"), start=("start", "min"), end=("end", "max"),
                                    grp=("grp", "first")).reset_index(drop=True)
print("Интервалов ChromHMM после объединения групп:", len(chm_blocks))

HIC_DIR = os.path.join(DATA, "hic"); os.makedirs(HIC_DIR, exist_ok=True)
HIC_SRC = {"CH12": "https://4dn-open-data-public.s3.amazonaws.com/fourfront-webprod/wfoutput/78844e4c-c98a-4692-acf2-6c4347199685/4DNFIA7GQD4X.mcool",
           "mESC": "https://4dn-open-data-public.s3.amazonaws.com/fourfront-webprod/wfoutput/c6181de2-2df3-4f17-a84a-3304c4c66511/4DNFIMN7XO4J.mcool"}
contacts = {}
if RUN_HIC:
    for name, url in HIC_SRC.items():
        pq = os.path.join(HIC_DIR, f"{name}_25kb_contacts.parquet")
        if not os.path.exists(pq):
            mc_path = os.path.join(HIC_DIR, f"{name}.mcool")
            sh(f"curl -sSfL --retry 5 -o {mc_path} {url}")
            hic_extract.extract_contacts(mc_path).to_parquet(pq)
            os.remove(mc_path)
        h = pd.read_parquet(pq); h["chr"] = h.chr.map(CI)
        contacts[name] = h
        print(name, "пар бинов с O/E>=2:", len(h))

## 6. Оригинальный MethyLasso на объединённых обучающих counts
MethyLasso запускается в режиме одного условия: counts всех обучающих образцов суммируются по CpG словаря
(псевдобалк; возраст не используется). Вход — шесть колонок `chr, start, end, meth, mC, uC`, координаты
0-based: `start = pos-1`, `end = pos+1`. С `--drop FALSE` сохраняется полная сегментация fused lasso,
включая неаннотированные сегменты; в эксперименте используются сегменты с >= 3 CpG словаря.
MethyLasso разработан для полногеномных данных; применение к RRBS — проверяемое допущение.

In [ ]:
ml_regions_table = None
if RUN_METHYLASSO:
    ml_dir = os.path.join(WORK, "methylasso_out"); os.makedirs(ml_dir, exist_ok=True)
    ml_tsv = os.path.join(ml_dir, "pooledtrain_lmr_umr_dmv.tsv")
    if not os.path.exists(ml_tsv):
        tr = R["train"]
        mc_sum = MC[tr].astype(np.int64).sum(0); cv_sum = CV[tr].astype(np.int64).sum(0)
        chn = np.array([f"chr{i+1}" for i in range(19)])[M.chr]
        pooled = pd.DataFrame({"chr": chn, "start": M.pos - 1, "end": M.pos + 1,
                               "meth": np.round(mc_sum / np.maximum(cv_sum, 1), 4), "mC": mc_sum, "uC": cv_sum - mc_sum})
        pooled_path = os.path.join(ml_dir, "pooled_train.tsv.gz")
        pooled[cv_sum > 0].to_csv(pooled_path, sep="\t", header=False, index=False)
        sh(f"cd {ml_dir} && Rscript {METHYLASSO_DIR}/MethyLasso.R --n1 pooledtrain --c1 {pooled_path} "
           f"-t 2 -f FALSE --drop FALSE > methylasso.log 2>&1")
    ml = pd.read_csv(ml_tsv, sep="\t")
    print("Категории MethyLasso:", ml.category.fillna("unannotated").value_counts().to_dict())
    ml = ml[ml.chr.isin(CI)].copy(); ml["chr"] = ml.chr.map(CI); ml["end"] = ml.end + 1
    ml_regions_table = ml.reset_index(drop=True)

## 7. Разбиения генома
Все разбиения строятся до обучения модели и не используют возраст:
DBSCAN (eps = 1000 п.н., minPts = 5) по координатам CpG словаря; окна 5 кб (RegBC);
сегменты MethyLasso; интервалы ChromHMM; пересечения DBSCAN ∩ MethyLasso и DBSCAN ∩ ChromHMM.

In [ ]:
allsites = np.arange(len(M.keys))
train_obs = (CV[R["train"]] >= MIN_COV).mean(0)
qc_sites = np.where(train_obs >= QC_FRAC_CPG)[0]
PART = {}
PART["DBSCAN"] = core.dbscan_regions(M.chr, M.pos, allsites, eps=DBSCAN_EPS, min_samples=DBSCAN_MINPTS)
PART["Window5kb"] = core.window_regions(M.chr, M.pos, allsites, width=5000)
if ml_regions_table is not None:
    PART["MethyLasso"] = core.interval_regions(M.chr, M.pos, allsites, ml_regions_table[["chr", "start", "end"]], min_cpg=3)
    PART["DBSCAN∩MethyLasso"] = core.intersect_partitions(PART["DBSCAN"][1], PART["MethyLasso"][1], M.chr, M.pos)
PART["ChromHMM"] = core.interval_regions(M.chr, M.pos, allsites, chm_blocks[["chr", "start", "end"]], min_cpg=3)
PART["DBSCAN∩ChromHMM"] = core.intersect_partitions(PART["DBSCAN"][1], PART["ChromHMM"][1], M.chr, M.pos)
# Для ветки C1: DBSCAN-кластеры, ограниченные CpG ветки A (тот же набор сайтов, что в ветке B)
mem_db = PART["DBSCAN"][1]
mem_same = np.where(np.isin(allsites, qc_sites), mem_db, -1)
PART["DBSCAN_sameSites"] = (PART["DBSCAN"][0], mem_same)
part_summary = pd.DataFrame([{"разбиение": k, "регионов": len(v[0]), "CpG в регионах": int((v[1] >= 0).sum()),
                              "медиана CpG/регион": float(np.median(np.bincount(v[1][v[1] >= 0])[np.bincount(v[1][v[1] >= 0]) > 0])),
                              "медиана длины, п.н.": float(v[0].eval("end-start").median())} for k, v in PART.items()])
print(part_summary.to_string(index=False))
part_summary.to_csv(os.path.join(OUT, "partitions_summary.tsv"), sep="\t", index=False)

## 8. Признаки и общая процедура оценки
Для каждого образца признаки региона вычисляются по его собственным CpG с покрытием >= 5:
`mean` — среднее β; `sd` — стандартное отклонение между CpG; `slope` — наклон β по нормированной
координате внутри региона; `enddiff` — среднее последней трети минус среднее первой трети; `bins4` —
средние четырёх последовательных бинов (контроль числа признаков для 4-компонентного описания).
Модель для всех представлений одна: импутация медианой train → стандартизация по train → LASSO с
alpha, выбранным 5-кратной CV на train. Внутренний тест, CR и внешние когорты в настройке не участвуют.

In [ ]:
ALL = np.arange(len(M.ids))
FEAT_CACHE = {}
def feats(part, kinds, mc=None, cov=None, rows=None):
    rows = ALL if rows is None else rows
    F = core.region_features(M, rows, PART[part][1], kinds=tuple(k for k in kinds if k != "bins4") +
                             (("bins",) if "bins4" in kinds else ()), nbins=4, mc=mc, cov=cov)
    F["bins4"] = F.pop("bins") if "bins" in F else None
    return np.hstack([F[k] for k in kinds])

def make_X(rep, mc=None, cov=None, rows=None):
    kind, part, kinds = REPS[rep]
    if kind == "cpg":
        return core.cpg_features(M, ALL if rows is None else rows, part, mc=mc, cov=cov)
    return feats(part, kinds, mc=mc, cov=cov, rows=rows)

FULL4 = ("mean", "sd", "slope", "enddiff")
in_db = qc_sites[mem_db[qc_sites] >= 0]
REPS = {
    # E1: отбор сайтов против агрегации
    "A_CpG_all_QC":            ("cpg", qc_sites, None),
    "B_CpG_in_DBSCAN":         ("cpg", in_db, None),
    "C1_DBSCAN_mean_sameSites": ("reg", "DBSCAN_sameSites", ("mean",)),
    "C2_DBSCAN_mean":          ("reg", "DBSCAN", ("mean",)),
    "D_DBSCAN_full4":          ("reg", "DBSCAN", FULL4),
    # E2: абляция описания паттерна на тех же DBSCAN-регионах
    "E2_DBSCAN_mean+sd":       ("reg", "DBSCAN", ("mean", "sd")),
    "E2_DBSCAN_mean+slope":    ("reg", "DBSCAN", ("mean", "slope")),
    "E2_DBSCAN_bins4":         ("reg", "DBSCAN", ("bins4",)),
    # E3: границы по сигналу
    "E3_Window5kb_mean":       ("reg", "Window5kb", ("mean",)),
    "E3_MethyLasso_mean":      ("reg", "MethyLasso", ("mean",)),
    "E3_MethyLasso_full4":     ("reg", "MethyLasso", FULL4),
    "E3_DBSCAN∩MethyLasso_mean":  ("reg", "DBSCAN∩MethyLasso", ("mean",)),
    "E3_DBSCAN∩MethyLasso_full4": ("reg", "DBSCAN∩MethyLasso", FULL4),
    # E4: границы состояний хроматина
    "E4_ChromHMM_mean":        ("reg", "ChromHMM", ("mean",)),
    "E4_DBSCAN∩ChromHMM_mean": ("reg", "DBSCAN∩ChromHMM", ("mean",)),
    "E4_DBSCAN∩ChromHMM_full4": ("reg", "DBSCAN∩ChromHMM", FULL4),
}
REPS = {k: v for k, v in REPS.items() if v[0] == "cpg" or v[1] in PART}

MODELS, PRED, ROWS = {}, {}, []
def evaluate(rep, X=None):
    t0 = time.time()
    X = make_X(rep) if X is None else X
    m = core.AgeModel(seed=SEED).fit(X[R["train"]], y[R["train"]])
    MODELS[rep] = m
    for s in SPLITS[1:]:
        p = m.predict(X[R[s]]); PRED[(rep, s)] = p
        if s == "CR":
            continue
        row = {"representation": rep, "split": s, **core.metrics(y[R[s]], p, seed=SEED),
               "n_features": m.n_features, "n_selected": m.n_selected,
               "missing_in_selected": m.feature_missing(X[R[s]])}
        ROWS.append(row)
    print(f"{rep:32s} признаков {m.n_features:>8d}, выбрано {m.n_selected:>4d}, {time.time()-t0:5.0f} с | " +
          " ".join(f"{s}: MAE {r['MAE']:.2f}" for r in ROWS[-3:] for s in [r['split']]), flush=True)
    del X

for rep in REPS:
    evaluate(rep)
res = pd.DataFrame(ROWS)
res.to_csv(os.path.join(OUT, "E1-E4_metrics.tsv"), sep="\t", index=False)

### Результаты E1–E4
Таблица: MAE (мес.) с 95% бутстреп-интервалом, R², смещение, число признаков, число выбранных LASSO
признаков и доля пропусков среди выбранных признаков в соответствующей когорте (до импутации).

In [ ]:
show = res.pivot_table(index="representation", columns="split", values=["MAE", "R2"]).round(2)
show = show.reindex(list(REPS))
print(show.to_string())
CONTRASTS = [  # (что сравниваем, лучше/хуже)
    ("E1 отбор сайтов: B против A", "B_CpG_in_DBSCAN", "A_CpG_all_QC"),
    ("E1 агрегация на тех же сайтах: C1 против B", "C1_DBSCAN_mean_sameSites", "B_CpG_in_DBSCAN"),
    ("E1 все наблюдаемые CpG кластера: C2 против C1", "C2_DBSCAN_mean", "C1_DBSCAN_mean_sameSites"),
    ("E1/E2 паттерн: D против C2", "D_DBSCAN_full4", "C2_DBSCAN_mean"),
    ("E2 контроль размерности: full4 против bins4", "D_DBSCAN_full4", "E2_DBSCAN_bins4"),
    ("E3 MethyLasso против DBSCAN (mean)", "E3_MethyLasso_mean", "C2_DBSCAN_mean"),
    ("E3 DBSCAN∩MethyLasso против DBSCAN (mean)", "E3_DBSCAN∩MethyLasso_mean", "C2_DBSCAN_mean"),
    ("E3 окна 5 кб против DBSCAN (mean)", "E3_Window5kb_mean", "C2_DBSCAN_mean"),
    ("E4 ChromHMM против DBSCAN (mean)", "E4_ChromHMM_mean", "C2_DBSCAN_mean"),
    ("E4 DBSCAN∩ChromHMM против DBSCAN (mean)", "E4_DBSCAN∩ChromHMM_mean", "C2_DBSCAN_mean"),
    ("E4 DBSCAN∩ChromHMM: паттерн против mean", "E4_DBSCAN∩ChromHMM_full4", "E4_DBSCAN∩ChromHMM_mean"),
]
crow = []
for name, a, b in CONTRASTS:
    if (a, "external_B6") not in PRED or (b, "external_B6") not in PRED:
        continue
    for s in ["internal_test", "external_B6", "external_BALB"]:
        d, lo, hi = core.paired_mae_diff(y[R[s]], PRED[(a, s)], PRED[(b, s)], seed=SEED)
        crow.append({"сравнение": name, "split": s, "ΔMAE (первый − второй)": d, "95% ДИ низ": lo, "95% ДИ верх": hi})
contr = pd.DataFrame(crow)
contr.to_csv(os.path.join(OUT, "E1-E4_paired_contrasts.tsv"), sep="\t", index=False)
print(contr.round(2).to_string(index=False))

## 9. E5 — пространственный контекст через графовый штраф
Узлы — DBSCAN-регионы (признак `mean`). Чтобы графовая модель была вычислимой, внутри train отбираются
10 000 регионов с наибольшей |r| с возрастом (одинаково для всех графов). Модель:
\[ \min_\theta \tfrac{1}{2n}\big(\|y-Z\theta\|^2+\lambda_2\textstyle\sum_{(r,q)\in E} w_{rq}(\theta_r-\theta_q)^2\big)+\alpha\|\theta\|_1, \]
где \(w_{rq}=\log_2(O/E)\). \(\lambda_2\in\{0,1,10,100\}\) и \(\alpha\) выбираются 5-кратной CV на train.
Графы: линейное соседство (соседние отобранные регионы ближе 100 кб), контакты CH12, CH12 с
рандомизацией рёбер при сохранении степеней узлов, контакты mESC (другой тип клеток).

In [ ]:
E5_ROWS, E5_INFO = [], []
if RUN_HIC:
    X5 = make_X("C2_DBSCAN_mean")
    pre5 = core.Preprocessor().fit(X5[R["train"]])
    Ztr = pre5.transform(X5[R["train"]]).astype(np.float64)
    ytr = y[R["train"]]
    r_age = np.abs(((Ztr - Ztr.mean(0)) * (ytr - ytr.mean())[:, None]).mean(0) / (Ztr.std(0) * ytr.std() + 1e-12))
    K = min(10000, Ztr.shape[1])
    top = np.argsort(-r_age)[:K]
    reg_db = PART["DBSCAN"][0]
    kept_regions = np.where(pre5.keep)[0][top]
    graphs = {"none": pd.DataFrame(columns=["a", "b", "w", "chr"]),
              "linear_100kb": core.linear_edges(reg_db, kept_regions),
              "CH12_contacts": core.contact_edges(reg_db, contacts["CH12"], nodes=kept_regions)}
    graphs["CH12_rewired"] = core.rewire_edges(graphs["CH12_contacts"], seed=SEED)
    graphs["mESC_contacts"] = core.contact_edges(reg_db, contacts["mESC"], nodes=kept_regions)
    Zs = {s: pre5.transform(X5[R[s]]).astype(np.float64)[:, top] for s in SPLITS[1:]}
    for gname, E in graphs.items():
        t0 = time.time()
        B = core.incidence(E, K)
        fit = core.graph_lasso_cv(Ztr[:, top], ytr, B, [0.0] if gname == "none" else [1.0, 10.0, 100.0], seed=SEED)
        E5_INFO.append({"graph": gname, "edges": len(E), "lambda2": fit["lam2"], "alpha": fit["alpha"],
                        "cv_MSE": fit["cv_mse"], "n_selected": int((fit["coef"] != 0).sum())})
        for s in SPLITS[1:]:
            p = (Zs[s] - fit["mu"]) @ fit["coef"] + fit["ymu"]
            PRED[("E5_" + gname, s)] = p
            if s != "CR":
                E5_ROWS.append({"graph": gname, "split": s, **core.metrics(y[R[s]], p, seed=SEED)})
        print(f"{gname:15s} рёбер {len(E):>7d} λ2={fit['lam2']:<6} выбрано {E5_INFO[-1]['n_selected']:>4d} "
              f"CV-MSE {fit['cv_mse']:.2f} {time.time()-t0:.0f} с", flush=True)
    e5 = pd.DataFrame(E5_ROWS); e5i = pd.DataFrame(E5_INFO)
    e5.to_csv(os.path.join(OUT, "E5_graph_metrics.tsv"), sep="\t", index=False)
    e5i.to_csv(os.path.join(OUT, "E5_graph_selection.tsv"), sep="\t", index=False)
    print(e5i.round(3).to_string(index=False))
    print(e5.pivot_table(index="graph", columns="split", values=["MAE", "R2"]).round(2).to_string())
    crow5 = []
    for g in ["linear_100kb", "CH12_contacts", "CH12_rewired", "mESC_contacts"]:
        for ref in ["none", "CH12_rewired"]:
            if g == ref:
                continue
            for s in ["internal_test", "external_B6"]:
                d, lo, hi = core.paired_mae_diff(y[R[s]], PRED[("E5_" + g, s)], PRED[("E5_" + ref, s)], seed=SEED)
                crow5.append({"граф": g, "против": ref, "split": s, "ΔMAE": d, "ДИ низ": lo, "ДИ верх": hi})
    e5c = pd.DataFrame(crow5); e5c.to_csv(os.path.join(OUT, "E5_paired_contrasts.tsv"), sep="\t", index=False)
    print(e5c.round(2).to_string(index=False))
    del X5

## 10. E6 — устойчивость к неполному наблюдению (внешняя когорта C57BL/6J)
Модели, обученные выше, не переобучаются. Для образцов внешнего теста counts ухудшаются двумя
разными способами: (а) случайное удаление доли наблюдаемых CpG (имитация несовпадения захваченных
сайтов); (б) биномиальное прореживание чтений (имитация меньшей глубины; приближение, т.к. чтения,
покрывающие несколько CpG, прореживаются независимо). Повторы с двумя зёрнами.

In [ ]:
E6_REPS = [r for r in ["A_CpG_all_QC", "C2_DBSCAN_mean", "D_DBSCAN_full4", "E3_Window5kb_mean",
                       "E3_MethyLasso_mean", "E4_DBSCAN∩ChromHMM_mean"] if r in MODELS]
LEVELS = [("mask", 0.25), ("mask", 0.5), ("mask", 0.75), ("thin", 0.5), ("thin", 0.25), ("thin", 0.1)]
rows_ext = R["external_B6"]
E6 = []
for rep in E6_REPS:
    E6.append({"representation": rep, "mode": "none", "level": 0, "seed": 0,
               **core.metrics(y[rows_ext], PRED[(rep, "external_B6")], n_boot=200)})
for mode, lev in LEVELS:
    for sd in range(2):
        rng = np.random.default_rng(SEED + sd)
        mc_d, cv_d = core.degrade_counts(MC[rows_ext], CV[rows_ext], mode, lev, rng)
        obs_frac = float((cv_d >= MIN_COV).mean())
        for rep in E6_REPS:
            X = make_X(rep, mc=mc_d, cov=cv_d, rows=np.arange(len(rows_ext)))
            p = MODELS[rep].predict(X)
            E6.append({"representation": rep, "mode": mode, "level": lev, "seed": sd, "obs_frac": obs_frac,
                       **core.metrics(y[rows_ext], p, n_boot=200)})
        print(f"{mode} {lev} seed {sd}: доля наблюдаемых CpG {obs_frac:.3f}", flush=True)
e6 = pd.DataFrame(E6)
e6.to_csv(os.path.join(OUT, "E6_robustness.tsv"), sep="\t", index=False)
e6s = e6.groupby(["representation", "mode", "level"])[["MAE", "R2"]].mean().round(2).unstack(["mode", "level"])
print(e6s.to_string())

## 11. Ускорение эпигенетического возраста при ограничении калорий (CR)
Остатки предсказаний CR-мышей относительно линии `pred ~ age`, построенной на контролях внутреннего
теста (они не участвовали в обучении). Отрицательное значение — замедление; p — критерий Манна–Уитни.

In [ ]:
cr_rows = []
for rep in list(MODELS) + [k[0] for k in PRED if k[0].startswith("E5_") and k[1] == "CR"]:
    if (rep, "CR") not in PRED:
        continue
    a = core.age_acceleration(PRED[(rep, "internal_test")], y[R["internal_test"]], PRED[(rep, "CR")], y[R["CR"]])
    cr_rows.append({"representation": rep, **a})
cr = pd.DataFrame(cr_rows).drop_duplicates("representation")
cr.to_csv(os.path.join(OUT, "CR_age_acceleration.tsv"), sep="\t", index=False)
print(cr.round(3).to_string(index=False))

## 12. Графики и сохранение результатов

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 6.5))
for k, s in enumerate(["internal_test", "external_B6"]):
    d = res[res.split == s].set_index("representation").reindex(list(REPS)).dropna(subset=["MAE"])
    ax[k].barh(range(len(d)), d.MAE, xerr=[d.MAE - d.MAE_lo, d.MAE_hi - d.MAE], color="#4C72B0", alpha=.85)
    ax[k].set_yticks(range(len(d))); ax[k].set_yticklabels(d.index, fontsize=8); ax[k].invert_yaxis()
    ax[k].set_xlabel("MAE, мес. (95% бутстреп-ДИ)"); ax[k].set_title({"internal_test": "Внутренний тест (Petkovich)",
                                                                       "external_B6": "Внешний тест (Thompson, C57BL/6J)"}[s])
plt.tight_layout(); plt.savefig(os.path.join(OUT, "fig_E1-E4_MAE.png"), dpi=150); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
for k, mode in enumerate(["mask", "thin"]):
    for rep in E6_REPS:
        d = e6[(e6.representation == rep) & (e6["mode"].isin(["none", mode]))].groupby("level").MAE.mean()
        x = d.index.values if mode == "mask" else np.where(d.index.values == 0, 1.0, d.index.values)
        o = np.argsort(x)
        ax[k].plot(x[o], d.values[o], "o-", label=rep)
    ax[k].set_xlabel("доля удалённых CpG" if mode == "mask" else "доля сохранённых чтений")
    if mode == "thin":
        ax[k].invert_xaxis()
    ax[k].set_ylabel("MAE, мес."); ax[k].set_title("Удаление CpG" if mode == "mask" else "Прореживание чтений")
ax[1].legend(fontsize=7, loc="upper left")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "fig_E6_robustness.png"), dpi=150); plt.show()

pd.DataFrame([{"representation": r, "sample_id": M.ids[i], "split": s, "age_months": y[i], "pred": p}
              for (r, s), pp in PRED.items() for i, p in zip(R[s], pp)]).to_csv(
    os.path.join(OUT, "predictions.tsv"), sep="\t", index=False)
for k, (reg, mem) in PART.items():
    if k == "DBSCAN_sameSites":
        continue
    out = reg.copy(); out["chr"] = "chr" + (out.chr + 1).astype(str); out["start"] = out.start - 1
    out[["chr", "start", "end", "n_cpg"]].to_csv(os.path.join(OUT, f"regions_{k}.bed"), sep="\t", header=False, index=False)
config = {k: v for k, v in dict(SEED=SEED, MIN_COV=MIN_COV, MIN_FRAC_TRAIN=MIN_FRAC_TRAIN, QC_FRAC_CPG=QC_FRAC_CPG,
          DBSCAN_EPS=DBSCAN_EPS, DBSCAN_MINPTS=DBSCAN_MINPTS, RUN_METHYLASSO=RUN_METHYLASSO, RUN_HIC=RUN_HIC,
          n_cpg_dictionary=int(len(keys))).items()}
if RUN_METHYLASSO and os.path.exists(os.path.join(METHYLASSO_DIR, "COMMIT")):
    config["methylasso_commit"] = open(os.path.join(METHYLASSO_DIR, "COMMIT")).read().strip()
json.dump(config, open(os.path.join(OUT, "config.json"), "w"), indent=2, ensure_ascii=False)
meta.to_csv(os.path.join(OUT, "samples_and_split.tsv"), sep="\t", index=False)
shutil.make_archive(os.path.join(WORK, "results_pipeline1"), "zip", OUT)
print("Архив:", os.path.join(WORK, "results_pipeline1.zip"))
if IN_COLAB:
    from google.colab import files
    files.download(os.path.join(WORK, "results_pipeline1.zip"))